In [1]:
#1 Install packages

%pip install -q numpy pandas matplotlib pillow opencv-python pyqt5
%matplotlib qt
%matplotlib tk
    

Note: you may need to restart the kernel to use updated packages.


In [3]:
#2 Imports and paths

from pathlib import Path
from zipfile import ZipFile
import re
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import rgb_to_hsv
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# Your dataset
ZIP_PATH = Path(r"C:\Users\rahma\Downloads\Dataset.zip")
EXTRACT_DIR = Path(r"C:\Users\rahma\Downloads\Dataset")

# All generated outputs will go here
PROJECT_ROOT = Path(
    r"C:\Users\rahma\Downloads\RINA_Internship_Analysis"
)

TRIAL_NAME = "First Trial (Two Trays)"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / TRIAL_NAME

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"
}

print("ZIP file:", ZIP_PATH)
print("Extracted dataset:", EXTRACT_DIR)
print("Output folder:", OUTPUT_ROOT)

ZIP file: C:\Users\rahma\Downloads\Dataset.zip
Extracted dataset: C:\Users\rahma\Downloads\Dataset
Output folder: C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)


In [5]:
#3 Extract the ZIP file

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Dataset ZIP file was not found:\n{ZIP_PATH}"
    )

if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(EXTRACT_DIR)

    print("Dataset extracted successfully.")
else:
    print("Dataset is already extracted.")

print("Dataset location:")
print(EXTRACT_DIR)

Dataset is already extracted.
Dataset location:
C:\Users\rahma\Downloads\Dataset


In [7]:
#4 Locate the First Trial folder

def clean_folder_name(text):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        text.lower()
    ).strip()


candidate_folders = [
    folder
    for folder in EXTRACT_DIR.rglob("*")
    if folder.is_dir()
    and clean_folder_name(folder.name) in {
        "first trial two trays",
        "first trial",
        "trial 1",
        "trial one",
    }
]

# Fallback search using known image names
if not candidate_folders:
    matching_images = [
        path
        for path in EXTRACT_DIR.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and "microbes" in path.name.lower()
        and "day 1" in path.name.lower()
    ]

    candidate_folders = sorted(
        {path.parent for path in matching_images}
    )


if not candidate_folders:
    raise FileNotFoundError(
        "Could not find the First Trial folder.\n"
        "The folder should contain files such as:\n"
        "Microbes (Day 1).jpg\n"
        "No Microbes (Day 1).jpg"
    )


def folder_score(folder):
    names = [
        path.name.lower()
        for path in folder.iterdir()
        if path.is_file()
    ]

    score = 0
    score += sum("microbes" in name for name in names)
    score += sum("no microbes" in name for name in names)
    score += sum("day 1" in name for name in names)
    score += sum("day 3" in name for name in names)
    score += sum("day 4" in name for name in names)

    return score


INPUT_DIR = max(
    candidate_folders,
    key=folder_score
)

print("First Trial folder found:")
print(INPUT_DIR)


First Trial folder found:
C:\Users\rahma\Downloads\Dataset\First Trial


In [9]:
#5 Identify the six Trial 1 images

def detect_treatment(filename):
    name = filename.lower()

    if (
        "no microbes" in name
        or "no_microbes" in name
        or "nomicrobes" in name
    ):
        return "No Microbes"

    if "microbes" in name or "microbe" in name:
        return "Microbes"

    return "Unknown"


def detect_day(filename):
    match = re.search(
        r"day[\s_\-\(\)]*([0-9]+)",
        filename.lower()
    )

    if match:
        return int(match.group(1))

    return None


image_paths = sorted(
    path
    for path in INPUT_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS
)


inventory_records = []

for path in image_paths:
    try:
        with Image.open(path) as image:
            width, height = image.size
    except Exception:
        width, height = None, None

    inventory_records.append(
        {
            "filename": path.name,
            "treatment": detect_treatment(path.name),
            "day": detect_day(path.name),
            "width": width,
            "height": height,
            "size_mb": round(
                path.stat().st_size / (1024 ** 2),
                3
            ),
            "path": str(path),
        }
    )


inventory_df = pd.DataFrame(inventory_records)

inventory_df = inventory_df.sort_values(
    ["treatment", "day", "filename"]
).reset_index(drop=True)


trial1_df = inventory_df[
    inventory_df["treatment"].isin(
        ["Microbes", "No Microbes"]
    )
    & inventory_df["day"].isin([1, 3, 4])
].copy()


display(inventory_df)


expected_images = {
    ("Microbes", 1),
    ("Microbes", 3),
    ("Microbes", 4),
    ("No Microbes", 1),
    ("No Microbes", 3),
    ("No Microbes", 4),
}

found_images = set(
    zip(
        trial1_df["treatment"],
        trial1_df["day"]
    )
)


if found_images == expected_images:
    print("All six Trial 1 images were found.")
else:
    print("Warning: expected images were not matched exactly.")
    print("Missing:", expected_images - found_images)
    print("Unexpected:", found_images - expected_images)


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

inventory_df.to_csv(
    OUTPUT_ROOT / "trial1_dataset_inventory.csv",
    index=False
)

,filename,treatment,day,width,height,size_mb,path
0,Microbes (Day 1).jpg,Microbes,1,4624,3472,6.455,C:\Users\rahma\Downloads\Dataset\First Trial\M...
1,Microbes (Day 3).jpg,Microbes,3,4624,3472,6.123,C:\Users\rahma\Downloads\Dataset\First Trial\M...
2,Microbes (Day 4).jpg,Microbes,4,4624,3472,5.476,C:\Users\rahma\Downloads\Dataset\First Trial\M...
3,No Microbes (Day 1).jpg,No Microbes,1,4624,3472,4.942,C:\Users\rahma\Downloads\Dataset\First Trial\N...
4,No Microbes (Day 3).jpg,No Microbes,3,4624,3472,4.266,C:\Users\rahma\Downloads\Dataset\First Trial\N...
5,No Microbes (Day 4).jpg,No Microbes,4,4624,3472,4.840,C:\Users\rahma\Downloads\Dataset\First Trial\N...


All six Trial 1 images were found.


In [11]:
#6 Create output folders

%matplotlib qt

# ============================================================
# 8
# TRIAL 1 OUTPUTS
# ============================================================

from pathlib import Path
import shutil
import re
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import rgb_to_hsv
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# These two variables must already exist from original Steps 3–7.
required_variables = ["trial1_df", "OUTPUT_ROOT"]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the original notebook Steps 3–7 first.\n"
        f"Missing variables: {missing_variables}"
    )

# Tray configuration
ROWS = 7
COLS = 10
TOTAL_CELLS = ROWS * COLS

CELL_SIZE = 120
STANDARD_WIDTH = COLS * CELL_SIZE
STANDARD_HEIGHT = ROWS * CELL_SIZE

# Rebuild all paths
STEP1_DIR = OUTPUT_ROOT / "01_standardized_images"
STANDARDIZED_DIR = STEP1_DIR / "standardized"
DIAGNOSTIC_DIR = STEP1_DIR / "diagnostics"

CLICK_CSV = STEP1_DIR / "clicked_corner_cell_centres.csv"
STANDARDIZED_SUMMARY_CSV = STEP1_DIR / "standardized_image_summary.csv"

STEP2_DIR = OUTPUT_ROOT / "02_germination_detection_strict"
OVERLAY_DIR = STEP2_DIR / "overlays"
MASK_DIR = STEP2_DIR / "green_masks"

CELL_CSV = STEP2_DIR / "cell_measurements_standardized_strict.csv"
SUMMARY_CSV = STEP2_DIR / "image_summary_standardized_strict.csv"
CUMULATIVE_CSV = (
    STEP2_DIR
    / "image_summary_standardized_strict_cumulative.csv"
)

STEP3_DIR = OUTPUT_ROOT / "03_charts_and_visual_comparisons"
CHART_DIR = STEP3_DIR / "charts"
VISUAL_DIR = STEP3_DIR / "visual_comparisons"

STEP4_DIR = OUTPUT_ROOT / "04_spatial_heatmaps"
STEP5_DIR = OUTPUT_ROOT / "05_descriptive_analysis"

# Delete all old incorrect analysis folders.
for folder in [
    STEP1_DIR,
    STEP2_DIR,
    STEP3_DIR,
    STEP4_DIR,
    STEP5_DIR,
]:
    if folder.exists():
        shutil.rmtree(folder)

# Recreate empty folders.
for folder in [
    STANDARDIZED_DIR,
    DIAGNOSTIC_DIR,
    OVERLAY_DIR,
    MASK_DIR,
    CHART_DIR,
    VISUAL_DIR,
    STEP4_DIR,
    STEP5_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("All incorrect Trial 1 outputs were removed.")
print("All six original images will be standardised again.")
print("Output location:")
print(OUTPUT_ROOT)


All incorrect Trial 1 outputs were removed.
All six original images will be standardised again.
Output location:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)


In [13]:
#7 Image standardisation functions

# ============================================================
#9
# PERSPECTIVE-STANDARDISATION FUNCTIONS
# ============================================================

def safe_name(text):
    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(text).lower()
    ).strip("_")

    return re.sub(r"_+", "_", text)


def read_image_bgr(path):
    path = Path(path)

    raw_data = np.fromfile(
        str(path),
        dtype=np.uint8
    )

    image = cv2.imdecode(
        raw_data,
        cv2.IMREAD_COLOR
    )

    if image is None:
        raise ValueError(
            f"Could not read image:\n{path}"
        )

    return image


def save_jpg_windows(
    path,
    image_bgr,
    quality=95
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    success, encoded = cv2.imencode(
        ".jpg",
        image_bgr,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            quality
        ]
    )

    if not success:
        raise ValueError(
            f"Could not save image:\n{path}"
        )

    encoded.tofile(str(path))


def order_four_points(points):
    """
    Return the points in this exact order:

    0 = top-left
    1 = top-right
    2 = bottom-right
    3 = bottom-left

    The user may click the four points in any order.
    """

    points = np.asarray(
        points,
        dtype=np.float32
    )

    if points.shape != (4, 2):
        raise ValueError(
            "Exactly four x-y points are required. "
            f"Received shape: {points.shape}"
        )

    # Check that four different points were selected.
    unique_points = np.unique(
        np.round(points, 2),
        axis=0
    )

    if len(unique_points) != 4:
        raise ValueError(
            "The same location appears to have been clicked more than once."
        )

    # Separate the two left points from the two right points.
    points_sorted_by_x = points[
        np.argsort(points[:, 0])
    ]

    left_points = points_sorted_by_x[:2]
    right_points = points_sorted_by_x[2:]

    # On each side, the smaller y-value is the top point.
    left_points = left_points[
        np.argsort(left_points[:, 1])
    ]

    right_points = right_points[
        np.argsort(right_points[:, 1])
    ]

    top_left = left_points[0]
    bottom_left = left_points[1]

    top_right = right_points[0]
    bottom_right = right_points[1]

    ordered = np.asarray(
        [
            top_left,
            top_right,
            bottom_right,
            bottom_left,
        ],
        dtype=np.float32
    )

    polygon_area = abs(
        cv2.contourArea(
            ordered.reshape(-1, 1, 2)
        )
    )

    if polygon_area < 1000:
        raise ValueError(
            "The four selected points do not form a valid tray rectangle."
        )

    return ordered


def collect_four_cell_centres(
    image_bgr,
    filename
):
    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    figure, axis = plt.subplots(
        figsize=(15, 10)
    )

    axis.imshow(image_rgb)

    axis.set_title(
        f"{filename}\n\n"
        "Click the CENTRES of these four extreme tray cells.\n"
        "You may click them in ANY order:\n\n"
        "• top-left tray cell\n"
        "• top-right tray cell\n"
        "• bottom-right tray cell\n"
        "• bottom-left tray cell\n\n"
        "Do not click the corners of the photograph."
    )

    axis.axis("on")
    plt.show(block=False)

    clicked_points = plt.ginput(
        4,
        timeout=0,
        show_clicks=True
    )

    plt.close(figure)

    if len(clicked_points) != 4:
        raise RuntimeError(
            f"Four clicks were required for {filename}, "
            f"but {len(clicked_points)} were recorded."
        )

    return order_four_points(
        clicked_points
    )


def bilinear_point(
    ordered_cell_centres,
    u,
    v
):
    top_left = ordered_cell_centres[0]
    top_right = ordered_cell_centres[1]
    bottom_right = ordered_cell_centres[2]
    bottom_left = ordered_cell_centres[3]

    top_position = (
        (1 - u) * top_left
        + u * top_right
    )

    bottom_position = (
        (1 - u) * bottom_left
        + u * bottom_right
    )

    return (
        (1 - v) * top_position
        + v * bottom_position
    )


def estimate_outer_grid_corners(
    ordered_cell_centres
):
    """
    The clicks are centres of the four extreme tray cells.

    Extend by half a cell to estimate the outer boundaries
    of the complete 7 × 10 cell grid.
    """

    ordered_cell_centres = order_four_points(
        ordered_cell_centres
    )

    u_left = -0.5 / (COLS - 1)
    u_right = 1 + 0.5 / (COLS - 1)

    v_top = -0.5 / (ROWS - 1)
    v_bottom = 1 + 0.5 / (ROWS - 1)

    outer_corners = np.asarray(
        [
            bilinear_point(
                ordered_cell_centres,
                u_left,
                v_top
            ),

            bilinear_point(
                ordered_cell_centres,
                u_right,
                v_top
            ),

            bilinear_point(
                ordered_cell_centres,
                u_right,
                v_bottom
            ),

            bilinear_point(
                ordered_cell_centres,
                u_left,
                v_bottom
            ),
        ],
        dtype=np.float32
    )

    return order_four_points(
        outer_corners
    )


def standardise_tray_image(
    image_bgr,
    cell_centres
):
    cell_centres = order_four_points(
        cell_centres
    )

    source_corners = estimate_outer_grid_corners(
        cell_centres
    )

    destination_corners = np.asarray(
        [
            [0, 0],

            [
                STANDARD_WIDTH - 1,
                0
            ],

            [
                STANDARD_WIDTH - 1,
                STANDARD_HEIGHT - 1
            ],

            [
                0,
                STANDARD_HEIGHT - 1
            ],
        ],
        dtype=np.float32
    )

    perspective_matrix = cv2.getPerspectiveTransform(
        source_corners,
        destination_corners
    )

    standardised_image = cv2.warpPerspective(
        image_bgr,
        perspective_matrix,
        (
            STANDARD_WIDTH,
            STANDARD_HEIGHT
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    return (
        standardised_image,
        source_corners
    )


def save_click_diagnostic(
    image_bgr,
    cell_centres,
    outer_corners,
    output_path
):
    diagnostic = image_bgr.copy()

    labels = [
        "TOP-LEFT",
        "TOP-RIGHT",
        "BOTTOM-RIGHT",
        "BOTTOM-LEFT",
    ]

    for point, label in zip(
        cell_centres,
        labels
    ):
        x = int(point[0])
        y = int(point[1])

        cv2.circle(
            diagnostic,
            (x, y),
            12,
            (0, 255, 255),
            -1
        )

        cv2.putText(
            diagnostic,
            label,
            (x + 12, y - 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2,
            cv2.LINE_AA
        )

    outer_polygon = outer_corners.reshape(
        (-1, 1, 2)
    ).astype(np.int32)

    cv2.polylines(
        diagnostic,
        [outer_polygon],
        True,
        (0, 255, 0),
        4
    )

    save_jpg_windows(
        output_path,
        diagnostic
    )


def save_grid_diagnostic(
    standardised_image,
    output_path
):
    diagnostic = standardised_image.copy()

    for column in range(COLS + 1):
        x = int(column * CELL_SIZE)

        cv2.line(
            diagnostic,
            (x, 0),
            (x, STANDARD_HEIGHT),
            (0, 255, 255),
            2
        )

    for row in range(ROWS + 1):
        y = int(row * CELL_SIZE)

        cv2.line(
            diagnostic,
            (0, y),
            (STANDARD_WIDTH, y),
            (0, 255, 255),
            2
        )

    for row in range(ROWS):
        for column in range(COLS):
            centre_x = int(
                column * CELL_SIZE
                + CELL_SIZE / 2
            )

            centre_y = int(
                row * CELL_SIZE
                + CELL_SIZE / 2
            )

            cv2.putText(
                diagnostic,
                f"{row + 1}-{column + 1}",
                (
                    centre_x - 22,
                    centre_y + 6
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

    save_jpg_windows(
        output_path,
        diagnostic
    )


print("Corrected standardisation functions loaded.")

Corrected standardisation functions loaded.


In [15]:
#8 Standardise all six Trial 1 images


if len(trial1_df) != 6:
    print(
        f"Warning: expected six Trial 1 images, "
        f"but found {len(trial1_df)}."
    )

standardisation_records = []
click_records = []

sorted_trial1_df = (
    trial1_df
    .sort_values(
        ["treatment", "day"]
    )
    .reset_index(drop=True)
)

print("Six image windows will open.")
print("Click four extreme tray-cell centres in each image.")
print("The click order does not matter.\n")

for image_number, row in sorted_trial1_df.iterrows():

    source_path = Path(
        row["path"]
    )

    treatment = row["treatment"]
    day = int(row["day"])

    print(
        f"Image {image_number + 1} "
        f"of {len(sorted_trial1_df)}:"
    )

    print(source_path.name)

    original_image = read_image_bgr(
        source_path
    )

    cell_centres = collect_four_cell_centres(
        original_image,
        source_path.name
    )

    standardised_image, outer_corners = (
        standardise_tray_image(
            original_image,
            cell_centres
        )
    )

    output_stem = (
        f"{safe_name(treatment)}"
        f"_day_{day:02d}"
    )

    standardised_path = (
        STANDARDIZED_DIR
        / f"{output_stem}_standardized.jpg"
    )

    click_diagnostic_path = (
        DIAGNOSTIC_DIR
        / f"{output_stem}_original_click_diagnostic.jpg"
    )

    grid_diagnostic_path = (
        DIAGNOSTIC_DIR
        / f"{output_stem}_standardized_grid_diagnostic.jpg"
    )

    save_jpg_windows(
        standardised_path,
        standardised_image
    )

    save_click_diagnostic(
        original_image,
        cell_centres,
        outer_corners,
        click_diagnostic_path
    )

    save_grid_diagnostic(
        standardised_image,
        grid_diagnostic_path
    )

    click_records.append(
        {
            "filename": source_path.name,

            "x_r1c1": cell_centres[0, 0],
            "y_r1c1": cell_centres[0, 1],

            "x_r1c10": cell_centres[1, 0],
            "y_r1c10": cell_centres[1, 1],

            "x_r7c10": cell_centres[2, 0],
            "y_r7c10": cell_centres[2, 1],

            "x_r7c1": cell_centres[3, 0],
            "y_r7c1": cell_centres[3, 1],
        }
    )

    standardisation_records.append(
        {
            "original_filename":
                source_path.name,

            "treatment":
                treatment,

            "day":
                day,

            "rows":
                ROWS,

            "columns":
                COLS,

            "total_cells":
                TOTAL_CELLS,

            "standardized_filename":
                standardised_path.name,

            "standardized_path":
                str(standardised_path),

            "original_click_diagnostic":
                click_diagnostic_path.name,

            "standardized_grid_diagnostic":
                grid_diagnostic_path.name,

            "output_width":
                STANDARD_WIDTH,

            "output_height":
                STANDARD_HEIGHT,
        }
    )

    print("Corrected standardised image saved.\n")


clicks_df = pd.DataFrame(
    click_records
)

clicks_df.to_csv(
    CLICK_CSV,
    index=False
)


standardised_summary_df = pd.DataFrame(
    standardisation_records
)

standardised_summary_df = (
    standardised_summary_df
    .sort_values(
        ["treatment", "day"]
    )
    .reset_index(drop=True)
)

standardised_summary_df.to_csv(
    STANDARDIZED_SUMMARY_CSV,
    index=False
)

display(standardised_summary_df)

print("All six images were standardised.")
print("Grid diagnostics:")
print(DIAGNOSTIC_DIR)

Six image windows will open.
Click four extreme tray-cell centres in each image.
The click order does not matter.

Image 1 of 6:
Microbes (Day 1).jpg
Corrected standardised image saved.

Image 2 of 6:
Microbes (Day 3).jpg
Corrected standardised image saved.

Image 3 of 6:
Microbes (Day 4).jpg
Corrected standardised image saved.

Image 4 of 6:
No Microbes (Day 1).jpg
Corrected standardised image saved.

Image 5 of 6:
No Microbes (Day 3).jpg
Corrected standardised image saved.

Image 6 of 6:
No Microbes (Day 4).jpg
Corrected standardised image saved.



,original_filename,treatment,day,rows,columns,total_cells,standardized_filename,standardized_path,original_click_diagnostic,standardized_grid_diagnostic,output_width,output_height
0,Microbes (Day 1).jpg,Microbes,1,7,10,70,microbes_day_01_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,microbes_day_01_original_click_diagnostic.jpg,microbes_day_01_standardized_grid_diagnostic.jpg,1200,840
1,Microbes (Day 3).jpg,Microbes,3,7,10,70,microbes_day_03_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,microbes_day_03_original_click_diagnostic.jpg,microbes_day_03_standardized_grid_diagnostic.jpg,1200,840
2,Microbes (Day 4).jpg,Microbes,4,7,10,70,microbes_day_04_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,microbes_day_04_original_click_diagnostic.jpg,microbes_day_04_standardized_grid_diagnostic.jpg,1200,840
3,No Microbes (Day 1).jpg,No Microbes,1,7,10,70,no_microbes_day_01_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,no_microbes_day_01_original_click_diagnostic.jpg,no_microbes_day_01_standardized_grid_diagnosti...,1200,840
4,No Microbes (Day 3).jpg,No Microbes,3,7,10,70,no_microbes_day_03_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,no_microbes_day_03_original_click_diagnostic.jpg,no_microbes_day_03_standardized_grid_diagnosti...,1200,840
5,No Microbes (Day 4).jpg,No Microbes,4,7,10,70,no_microbes_day_04_standardized.jpg,C:\Users\rahma\Downloads\RINA_Internship_Analy...,no_microbes_day_04_original_click_diagnostic.jpg,no_microbes_day_04_standardized_grid_diagnosti...,1200,840


All six images were standardised.
Grid diagnostics:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\01_standardized_images\diagnostics


In [17]:
#9 DISPLAY ALL SIX STANDARDISED GRID DIAGNOSTICS


grid_paths = sorted(
    DIAGNOSTIC_DIR.glob(
        "*standardized_grid_diagnostic.jpg"
    )
)

if len(grid_paths) != 6:
    print(
        f"Warning: expected six grid diagnostics, "
        f"but found {len(grid_paths)}."
    )

figure, axes = plt.subplots(
    3,
    2,
    figsize=(18, 18)
)

axes = axes.flatten()

for axis, path in zip(
    axes,
    grid_paths
):
    axis.imshow(
        Image.open(path).convert("RGB")
    )

    axis.set_title(path.name)
    axis.axis("off")

for remaining_axis in axes[
    len(grid_paths):
]:
    remaining_axis.axis("off")

plt.tight_layout()
plt.show()

print("STOP AND CHECK ALL SIX IMAGES.")
print("Continue only when every tray has:")
print("- no crossed diagonal sections")
print("- no large black centre")
print("- seven visible rows")
print("- ten visible columns")
print("- one tray cell inside each yellow box")

STOP AND CHECK ALL SIX IMAGES.
Continue only when every tray has:
- no crossed diagonal sections
- no large black centre
- seven visible rows
- ten visible columns
- one tray cell inside each yellow box


In [21]:
#10 Display the grid diagnostics

grid_diagnostic_paths = sorted(
    DIAGNOSTIC_DIR.glob(
        "*standardized_grid_diagnostic.jpg"
    )
)


figure, axes = plt.subplots(
    len(grid_diagnostic_paths),
    1,
    figsize=(
        14,
        7 * len(grid_diagnostic_paths)
    )
)


if len(grid_diagnostic_paths) == 1:
    axes = [axes]


for axis, path in zip(
    axes,
    grid_diagnostic_paths
):
    axis.imshow(
        Image.open(path).convert("RGB")
    )

    axis.set_title(path.name)
    axis.axis("off")


plt.tight_layout()
plt.show()

In [21]:
#11 STRICT VISIBLE-GREEN GERMINATION DETECTION

INNER_MARGIN_RATIO = 0.12

MIN_GREEN_PIXELS = 15
MIN_GREEN_RATIO = 0.0015
MIN_MEAN_GREEN_STRENGTH = 5.0
MIN_LARGEST_GREEN_CLUSTER = 7


def load_image_rgb(path):
    return np.asarray(
        Image.open(path).convert("RGB")
    )


def green_pixel_mask(roi):
    rgb = roi.astype(np.float32)

    hsv = rgb_to_hsv(
        rgb / 255.0
    )

    red = rgb[:, :, 0]
    green = rgb[:, :, 1]
    blue = rgb[:, :, 2]

    hue = hsv[:, :, 0]
    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]

    excess_green = (
        2 * green
        - red
        - blue
    )

    hsv_rule = (
        (hue >= 0.16)
        & (hue <= 0.42)
        & (saturation >= 0.18)
        & (value >= 0.18)
    )

    rgb_rule = (
        (green >= 45)
        & (green > red * 1.04)
        & (green > blue * 1.04)
        & ((green - red) >= 6)
        & (excess_green >= 8)
    )

    final_mask = (
        hsv_rule
        & rgb_rule
    )

    return (
        final_mask,
        excess_green
    )


def largest_connected_component_size(mask):
    binary = mask.astype(np.uint8)

    if binary.max() == 0:
        return 0

    component_count, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )
    )

    if component_count <= 1:
        return 0

    largest_size = stats[
        1:,
        cv2.CC_STAT_AREA
    ].max()

    return int(largest_size)


def analyse_cell(
    image,
    row_index,
    column_index
):
    image_height, image_width = image.shape[:2]

    cell_width = image_width / COLS
    cell_height = image_height / ROWS

    x1 = int(
        column_index * cell_width
    )

    x2 = int(
        (column_index + 1) * cell_width
    )

    y1 = int(
        row_index * cell_height
    )

    y2 = int(
        (row_index + 1) * cell_height
    )

    margin_x = int(
        cell_width * INNER_MARGIN_RATIO
    )

    margin_y = int(
        cell_height * INNER_MARGIN_RATIO
    )

    roi_x1 = x1 + margin_x
    roi_x2 = x2 - margin_x

    roi_y1 = y1 + margin_y
    roi_y2 = y2 - margin_y

    roi = image[
        roi_y1:roi_y2,
        roi_x1:roi_x2
    ]

    mask, green_strength = (
        green_pixel_mask(roi)
    )

    green_pixels = int(mask.sum())
    total_pixels = int(mask.size)

    green_ratio = (
        green_pixels / total_pixels
        if total_pixels > 0
        else 0.0
    )

    if green_pixels > 0:
        mean_green_strength = float(
            green_strength[mask].mean()
        )

        max_green_strength = float(
            green_strength[mask].max()
        )

    else:
        mean_green_strength = 0.0
        max_green_strength = 0.0

    largest_cluster = (
        largest_connected_component_size(
            mask
        )
    )

    germinated = (
        green_pixels
        >= MIN_GREEN_PIXELS

        and green_ratio
        >= MIN_GREEN_RATIO

        and mean_green_strength
        >= MIN_MEAN_GREEN_STRENGTH

        and largest_cluster
        >= MIN_LARGEST_GREEN_CLUSTER
    )

    return {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,

        "roi_x1": roi_x1,
        "roi_y1": roi_y1,
        "roi_x2": roi_x2,
        "roi_y2": roi_y2,

        "green_pixels":
            green_pixels,

        "total_pixels":
            total_pixels,

        "green_ratio":
            green_ratio,

        "mean_green_strength":
            mean_green_strength,

        "max_green_strength":
            max_green_strength,

        "largest_green_cluster":
            largest_cluster,

        "germinated_estimate":
            bool(germinated),
    }


def get_font(size):
    possible_fonts = [
        Path(
            r"C:\Windows\Fonts\arial.ttf"
        ),

        Path(
            r"C:\Windows\Fonts\calibri.ttf"
        ),
    ]

    for font_path in possible_fonts:
        if font_path.exists():
            return ImageFont.truetype(
                str(font_path),
                size
            )

    return ImageFont.load_default()


def create_green_mask_image(
    image,
    cell_records
):
    mask_image = np.zeros_like(image)

    for record in cell_records:
        x1 = int(record["roi_x1"])
        y1 = int(record["roi_y1"])

        x2 = int(record["roi_x2"])
        y2 = int(record["roi_y2"])

        roi = image[
            y1:y2,
            x1:x2
        ]

        mask, green_strength = (
            green_pixel_mask(roi)
        )

        mask_region = mask_image[
            y1:y2,
            x1:x2
        ]

        mask_region[mask] = [
            0,
            255,
            0
        ]

    return mask_image


def save_detection_overlay(
    image,
    cell_records,
    output_path,
    title_text
):
    pil_image = Image.fromarray(image)
    drawer = ImageDraw.Draw(pil_image)

    title_font = get_font(22)
    cell_font = get_font(14)

    for record in cell_records:

        if record["germinated_estimate"]:
            colour = "lime"
        else:
            colour = "red"

        x1 = int(record["x1"])
        y1 = int(record["y1"])

        x2 = int(record["x2"])
        y2 = int(record["y2"])

        roi_x1 = int(record["roi_x1"])
        roi_y1 = int(record["roi_y1"])

        roi_x2 = int(record["roi_x2"])
        roi_y2 = int(record["roi_y2"])

        drawer.rectangle(
            [x1, y1, x2, y2],
            outline="yellow",
            width=2
        )

        drawer.rectangle(
            [
                roi_x1,
                roi_y1,
                roi_x2,
                roi_y2
            ],
            outline=colour,
            width=3
        )

        drawer.text(
            (
                x1 + 5,
                y1 + 5
            ),
            f'{record["row"]}-{record["col"]}',
            fill="white",
            font=cell_font
        )

    title_height = 80

    canvas = Image.new(
        "RGB",
        (
            pil_image.width,
            pil_image.height + title_height
        ),
        "white"
    )

    canvas.paste(
        pil_image,
        (0, title_height)
    )

    title_drawer = ImageDraw.Draw(canvas)

    title_drawer.text(
        (20, 22),
        title_text,
        fill="black",
        font=title_font
    )

    canvas.save(
        output_path,
        quality=95
    )


print("Germination detection functions loaded.")

Germination detection functions loaded.


In [23]:
#12 RUN GERMINATION DETECTION FOR ALL SIX IMAGES

all_cell_records = []
image_summary_records = []

for _, image_row in (
    standardised_summary_df.iterrows()
):

    image_path = Path(
        image_row["standardized_path"]
    )

    treatment = image_row["treatment"]
    day = int(image_row["day"])

    print(
        f"Processing: {treatment}, Day {day}"
    )

    image = load_image_rgb(
        image_path
    )

    image_cell_records = []

    for row_index in range(ROWS):

        for column_index in range(COLS):

            result = analyse_cell(
                image,
                row_index,
                column_index
            )

            record = {
                "filename":
                    image_path.name,

                "treatment":
                    treatment,

                "day":
                    day,

                "row":
                    row_index + 1,

                "col":
                    column_index + 1,

                "cell_id":
                    (
                        f"R{row_index + 1:02d}"
                        f"_C{column_index + 1:02d}"
                    ),

                **result
            }

            image_cell_records.append(
                record
            )

            all_cell_records.append(
                record
            )

    germinated_count = sum(
        int(
            record["germinated_estimate"]
        )
        for record in image_cell_records
    )

    germination_percent = (
        germinated_count
        / TOTAL_CELLS
        * 100
    )

    output_base = safe_name(
        image_path.stem
    )

    overlay_path = (
        OVERLAY_DIR
        / f"{output_base}_strict_overlay.jpg"
    )

    mask_path = (
        MASK_DIR
        / f"{output_base}_strict_green_mask.jpg"
    )

    save_detection_overlay(
        image,
        image_cell_records,
        overlay_path,
        (
            f"{treatment} | Day {day} | "
            f"Estimated germination: "
            f"{germinated_count}/{TOTAL_CELLS} "
            f"({germination_percent:.2f}%)"
        )
    )

    green_mask_image = (
        create_green_mask_image(
            image,
            image_cell_records
        )
    )

    Image.fromarray(
        green_mask_image
    ).save(
        mask_path,
        quality=95
    )

    mean_green_ratio = float(
        np.mean(
            [
                record["green_ratio"]
                for record
                in image_cell_records
            ]
        )
    )

    image_summary_records.append(
        {
            "filename":
                image_path.name,

            "treatment":
                treatment,

            "day":
                day,

            "total_cells":
                TOTAL_CELLS,

            "estimated_germinated_cells":
                germinated_count,

            "estimated_not_germinated_cells":
                TOTAL_CELLS
                - germinated_count,

            "estimated_germination_percent":
                round(
                    germination_percent,
                    2
                ),

            "mean_green_ratio":
                round(
                    mean_green_ratio,
                    6
                ),

            "overlay_filename":
                overlay_path.name,

            "green_mask_filename":
                mask_path.name,
        }
    )


cell_df = pd.DataFrame(
    all_cell_records
)

summary_df = pd.DataFrame(
    image_summary_records
)

summary_df = (
    summary_df
    .sort_values(
        ["treatment", "day"]
    )
    .reset_index(drop=True)
)


# Create cumulative results.
cumulative_records = []

for treatment, treatment_group in (
    summary_df.groupby("treatment")
):

    previous_best = 0

    treatment_group = (
        treatment_group
        .sort_values("day")
    )

    for _, result_row in (
        treatment_group.iterrows()
    ):

        raw_count = int(
            result_row[
                "estimated_germinated_cells"
            ]
        )

        cumulative_count = max(
            previous_best,
            raw_count
        )

        cumulative_record = (
            result_row.to_dict()
        )

        cumulative_record[
            "count_decreased_before_correction"
        ] = (
            raw_count < previous_best
        )

        cumulative_record[
            "cumulative_germinated_cells"
        ] = cumulative_count

        cumulative_record[
            "cumulative_germination_percent"
        ] = round(
            cumulative_count
            / int(result_row["total_cells"])
            * 100,
            2
        )

        cumulative_records.append(
            cumulative_record
        )

        previous_best = cumulative_count


cumulative_df = pd.DataFrame(
    cumulative_records
)

cumulative_df = (
    cumulative_df
    .sort_values(
        ["treatment", "day"]
    )
    .reset_index(drop=True)
)


cell_df.to_csv(
    CELL_CSV,
    index=False
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False
)

cumulative_df.to_csv(
    CUMULATIVE_CSV,
    index=False
)


display(
    cumulative_df[
        [
            "filename",
            "treatment",
            "day",
            "total_cells",

            "estimated_germinated_cells",
            "estimated_germination_percent",

            "cumulative_germinated_cells",
            "cumulative_germination_percent",

            "count_decreased_before_correction",
        ]
    ]
)

print("Detection completed.")
print("Cell CSV:")
print(CELL_CSV)

print("Summary CSV:")
print(SUMMARY_CSV)

print("Cumulative CSV:")
print(CUMULATIVE_CSV)

Processing: Microbes, Day 1
Processing: Microbes, Day 3
Processing: Microbes, Day 4
Processing: No Microbes, Day 1
Processing: No Microbes, Day 3
Processing: No Microbes, Day 4


,filename,treatment,day,total_cells,estimated_germinated_cells,estimated_germination_percent,cumulative_germinated_cells,cumulative_germination_percent,count_decreased_before_correction
0,microbes_day_01_standardized.jpg,Microbes,1,70,26,37.14,26,37.14,False
1,microbes_day_03_standardized.jpg,Microbes,3,70,45,64.29,45,64.29,False
2,microbes_day_04_standardized.jpg,Microbes,4,70,50,71.43,50,71.43,False
3,no_microbes_day_01_standardized.jpg,No Microbes,1,70,37,52.86,37,52.86,False
4,no_microbes_day_03_standardized.jpg,No Microbes,3,70,48,68.57,48,68.57,False
5,no_microbes_day_04_standardized.jpg,No Microbes,4,70,54,77.14,54,77.14,False


Detection completed.
Cell CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\cell_measurements_standardized_strict.csv
Summary CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\image_summary_standardized_strict.csv
Cumulative CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\image_summary_standardized_strict_cumulative.csv


In [25]:
#13 DISPLAY ALL SIX DETECTION OVERLAYS

overlay_paths = sorted(
    OVERLAY_DIR.glob("*.jpg")
)

figure, axes = plt.subplots(
    3,
    2,
    figsize=(18, 20)
)

axes = axes.flatten()

for axis, overlay_path in zip(
    axes,
    overlay_paths
):
    axis.imshow(
        Image.open(
            overlay_path
        ).convert("RGB")
    )

    axis.set_title(
        overlay_path.name
    )

    axis.axis("off")

for remaining_axis in axes[
    len(overlay_paths):
]:
    remaining_axis.axis("off")

plt.tight_layout()
plt.show()

print("Green inner box = detected.")
print("Red inner box = not detected.")
print("Yellow outer box = full tray cell.")

Green inner box = detected.
Red inner box = not detected.
Yellow outer box = full tray cell.


In [27]:
#14 CREATE GERMINATION CHARTS

def save_line_chart(
    dataframe,
    value_column,
    y_label,
    title,
    output_filename,
    upper_limit=None
):
    plt.figure(figsize=(9, 6))

    for treatment, treatment_group in (
        dataframe.groupby("treatment")
    ):

        treatment_group = (
            treatment_group
            .sort_values("day")
        )

        plt.plot(
            treatment_group["day"],
            treatment_group[value_column],
            marker="o",
            linewidth=2,
            label=treatment
        )

    plt.title(title)
    plt.xlabel("Day")
    plt.ylabel(y_label)

    if upper_limit is not None:
        plt.ylim(
            0,
            upper_limit
        )

    plt.xticks(
        sorted(
            dataframe["day"].unique()
        )
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()
    plt.tight_layout()

    output_path = (
        CHART_DIR
        / output_filename
    )

    plt.savefig(
        output_path,
        dpi=300
    )

    plt.show()

    return output_path


save_line_chart(
    cumulative_df,

    "estimated_germination_percent",

    "Raw automated germination (%)",

    "Trial 1: Raw Automated Germination Percentage",

    "first_trial_raw_automated_germination_percentage.png",

    105
)


save_line_chart(
    cumulative_df,

    "cumulative_germination_percent",

    "Cumulative estimated germination (%)",

    "Trial 1: Cumulative Germination Percentage",

    "first_trial_cumulative_germination_percentage.png",

    105
)


save_line_chart(
    cumulative_df,

    "cumulative_germinated_cells",

    "Cumulative germinated cells",

    "Trial 1: Cumulative Germinated Cell Count",

    "first_trial_cumulative_germinated_cells.png",

    TOTAL_CELLS + 5
)


final_result_df = (
    cumulative_df
    .sort_values("day")
    .groupby(
        "treatment",
        as_index=False
    )
    .tail(1)
    .sort_values("treatment")
)


plt.figure(figsize=(8, 6))

bars = plt.bar(
    final_result_df["treatment"],
    final_result_df[
        "cumulative_germination_percent"
    ]
)

plt.title(
    "Trial 1: Final Germination Comparison"
)

plt.xlabel("Treatment")

plt.ylabel(
    "Final cumulative germination (%)"
)

plt.ylim(0, 105)

plt.grid(
    axis="y",
    alpha=0.3
)

for bar, value in zip(
    bars,
    final_result_df[
        "cumulative_germination_percent"
    ]
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,

        value + 2,

        f"{value:.2f}%",

        ha="center"
    )

plt.tight_layout()

plt.savefig(
    CHART_DIR
    / (
        "first_trial_final_"
        "microbes_vs_no_microbes_comparison.png"
    ),
    dpi=300
)

plt.show()

display(final_result_df)

,filename,treatment,day,total_cells,estimated_germinated_cells,estimated_not_germinated_cells,estimated_germination_percent,mean_green_ratio,overlay_filename,green_mask_filename,count_decreased_before_correction,cumulative_germinated_cells,cumulative_germination_percent
2,microbes_day_04_standardized.jpg,Microbes,4,70,50,20,71.43,0.011437,microbes_day_04_standardized_strict_overlay.jpg,microbes_day_04_standardized_strict_green_mask...,False,50,71.43
5,no_microbes_day_04_standardized.jpg,No Microbes,4,70,54,16,77.14,0.020694,no_microbes_day_04_standardized_strict_overlay...,no_microbes_day_04_standardized_strict_green_m...,False,54,77.14


In [29]:
#15 CREATE TREATMENT VISUAL COMPARISONS

def get_result_row(
    treatment,
    day
):
    matched = cumulative_df[
        (
            cumulative_df["treatment"]
            == treatment
        )
        &
        (
            cumulative_df["day"]
            == day
        )
    ]

    if matched.empty:
        return None

    return matched.iloc[0]


def result_description(result_row):
    return (
        f'Raw: '
        f'{int(result_row["estimated_germinated_cells"])}'
        f'/{TOTAL_CELLS} | '
        f'Cumulative: '
        f'{int(result_row["cumulative_germinated_cells"])}'
        f'/{TOTAL_CELLS} '
        f'('
        f'{float(result_row["cumulative_germination_percent"]):.2f}%'
        f')'
    )


for day in sorted(
    cumulative_df["day"].unique()
):

    microbes_row = get_result_row(
        "Microbes",
        day
    )

    no_microbes_row = get_result_row(
        "No Microbes",
        day
    )

    if (
        microbes_row is None
        or no_microbes_row is None
    ):
        continue

    comparison_sets = [
        (
            [
                STANDARDIZED_DIR
                / microbes_row["filename"],

                STANDARDIZED_DIR
                / no_microbes_row["filename"],
            ],

            "Standardised Image",

            (
                f"first_trial_day_{day:02d}_"
                "standardized_microbes_vs_no_microbes.png"
            )
        ),

        (
            [
                OVERLAY_DIR
                / microbes_row["overlay_filename"],

                OVERLAY_DIR
                / no_microbes_row["overlay_filename"],
            ],

            "Strict Detection Overlay",

            (
                f"first_trial_day_{day:02d}_"
                "strict_overlay_microbes_vs_no_microbes.png"
            )
        ),

        (
            [
                MASK_DIR
                / microbes_row["green_mask_filename"],

                MASK_DIR
                / no_microbes_row["green_mask_filename"],
            ],

            "Strict Green Mask",

            (
                f"first_trial_day_{day:02d}_"
                "strict_green_mask_microbes_vs_no_microbes.png"
            )
        ),
    ]

    for (
        image_paths,
        comparison_name,
        output_filename
    ) in comparison_sets:

        figure, axes = plt.subplots(
            1,
            2,
            figsize=(16, 8)
        )

        for axis, image_path, result_row in zip(
            axes,
            image_paths,
            [
                microbes_row,
                no_microbes_row
            ]
        ):
            axis.imshow(
                Image.open(
                    image_path
                ).convert("RGB")
            )

            axis.set_title(
                f'{result_row["treatment"]} '
                f'— Day {day}\n'
                f'{result_description(result_row)}'
            )

            axis.axis("off")

        figure.suptitle(
            f"Trial 1 {comparison_name} "
            f"Comparison — Day {day}",
            fontsize=16
        )

        plt.tight_layout()

        plt.savefig(
            VISUAL_DIR
            / output_filename,
            dpi=300
        )

        plt.show()


report_table = cumulative_df[
    [
        "treatment",
        "day",
        "total_cells",
        "estimated_germinated_cells",
        "estimated_germination_percent",
        "cumulative_germinated_cells",
        "cumulative_germination_percent",
        "count_decreased_before_correction",
    ]
].copy()


report_table.to_csv(
    STEP3_DIR
    / "first_trial_report_table_strict_standardized.csv",
    index=False
)

display(report_table)

,treatment,day,total_cells,estimated_germinated_cells,estimated_germination_percent,cumulative_germinated_cells,cumulative_germination_percent,count_decreased_before_correction
0,Microbes,1,70,26,37.14,26,37.14,False
1,Microbes,3,70,45,64.29,45,64.29,False
2,Microbes,4,70,50,71.43,50,71.43,False
3,No Microbes,1,70,37,52.86,37,52.86,False
4,No Microbes,3,70,48,68.57,48,68.57,False
5,No Microbes,4,70,54,77.14,54,77.14,False


In [33]:
#16 Create cumulative germination results

cumulative_records = []


for treatment, group in (
    summary_df.groupby("treatment")
):

    previous_best = 0


    group = group.sort_values("day")


    for _, row in group.iterrows():

        raw_count = int(
            row[
                "estimated_germinated_cells"
            ]
        )


        cumulative_count = max(
            previous_best,
            raw_count
        )


        row_record = row.to_dict()


        row_record[
            "count_decreased_before_correction"
        ] = (
            raw_count
            < previous_best
        )


        row_record[
            "cumulative_germinated_cells"
        ] = cumulative_count


        row_record[
            "cumulative_germination_percent"
        ] = round(
            cumulative_count
            / int(row["total_cells"])
            * 100,
            2
        )


        cumulative_records.append(
            row_record
        )


        previous_best = (
            cumulative_count
        )


cumulative_df = pd.DataFrame(
    cumulative_records
)


cumulative_df = cumulative_df.sort_values(
    ["treatment", "day"]
).reset_index(drop=True)


cell_df.to_csv(
    CELL_CSV,
    index=False
)


summary_df.to_csv(
    SUMMARY_CSV,
    index=False
)


cumulative_df.to_csv(
    CUMULATIVE_CSV,
    index=False
)


display(
    cumulative_df[
        [
            "filename",
            "treatment",
            "day",
            "total_cells",

            "estimated_germinated_cells",
            "estimated_germination_percent",

            "cumulative_germinated_cells",
            "cumulative_germination_percent",

            "count_decreased_before_correction"
        ]
    ]
)


print("Cell-level CSV:")
print(CELL_CSV)

print("\nSummary CSV:")
print(SUMMARY_CSV)

print("\nCumulative CSV:")
print(CUMULATIVE_CSV)

,filename,treatment,day,total_cells,estimated_germinated_cells,estimated_germination_percent,cumulative_germinated_cells,cumulative_germination_percent,count_decreased_before_correction
0,microbes_day_01_standardized.jpg,Microbes,1,70,0,0.0,0,0.0,False
1,microbes_day_03_standardized.jpg,Microbes,3,70,0,0.0,0,0.0,False
2,microbes_day_04_standardized.jpg,Microbes,4,70,0,0.0,0,0.0,False
3,no_microbes_day_01_standardized.jpg,No Microbes,1,70,0,0.0,0,0.0,False
4,no_microbes_day_03_standardized.jpg,No Microbes,3,70,0,0.0,0,0.0,False
5,no_microbes_day_04_standardized.jpg,No Microbes,4,70,0,0.0,0,0.0,False


Cell-level CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\cell_measurements_standardized_strict.csv

Summary CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\image_summary_standardized_strict.csv

Cumulative CSV:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)\02_germination_detection_strict\image_summary_standardized_strict_cumulative.csv


In [30]:
#16 CREATE SPATIAL HEATMAPS

def boolean_to_integer(value):

    if isinstance(
        value,
        (bool, np.bool_)
    ):
        return int(value)

    return int(
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
            "y"
        }
    )


def create_germination_grid(
    treatment_day_group
):
    grid = np.zeros(
        (
            ROWS,
            COLS
        ),
        dtype=int
    )

    for _, cell_row in (
        treatment_day_group.iterrows()
    ):

        row_index = (
            int(cell_row["row"])
            - 1
        )

        column_index = (
            int(cell_row["col"])
            - 1
        )

        grid[
            row_index,
            column_index
        ] = boolean_to_integer(
            cell_row[
                "germinated_estimate"
            ]
        )

    return grid


for (
    treatment,
    day
), treatment_day_group in cell_df.groupby(
    [
        "treatment",
        "day"
    ]
):

    grid = create_germination_grid(
        treatment_day_group
    )

    germinated_count = int(
        grid.sum()
    )

    germination_percent = (
        germinated_count
        / TOTAL_CELLS
        * 100
    )

    plt.figure(figsize=(10, 7))

    plt.imshow(
        grid,
        vmin=0,
        vmax=1
    )

    for row_index in range(ROWS):

        for column_index in range(COLS):

            label = (
                "G"
                if grid[
                    row_index,
                    column_index
                ] == 1

                else "-"
            )

            plt.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                fontsize=9
            )

    plt.title(
        "Trial 1 Spatial Germination Heatmap\n"
        f"{treatment} — Day {int(day)} | "
        f"{germinated_count}/{TOTAL_CELLS} "
        f"cells ({germination_percent:.1f}%)"
    )

    plt.xlabel("Column")
    plt.ylabel("Row")

    plt.xticks(
        range(COLS),
        range(1, COLS + 1)
    )

    plt.yticks(
        range(ROWS),
        range(1, ROWS + 1)
    )

    plt.colorbar(
        label=(
            "0 = not detected, "
            "1 = germinated"
        )
    )

    plt.tight_layout()

    output_path = (
        STEP4_DIR
        / (
            f"first_trial_"
            f"{safe_name(treatment)}"
            f"_day_{int(day):02d}"
            "_spatial_heatmap.png"
        )
    )

    plt.savefig(
        output_path,
        dpi=300
    )

    plt.show()


# Side-by-side heatmap comparison for each day.
for day in sorted(
    cell_df["day"].unique()
):

    day_data = cell_df[
        cell_df["day"] == day
    ]

    microbes_data = day_data[
        day_data["treatment"]
        == "Microbes"
    ]

    no_microbes_data = day_data[
        day_data["treatment"]
        == "No Microbes"
    ]

    if (
        microbes_data.empty
        or no_microbes_data.empty
    ):
        continue

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(16, 7)
    )

    for axis, group, treatment in [
        (
            axes[0],
            microbes_data,
            "Microbes"
        ),

        (
            axes[1],
            no_microbes_data,
            "No Microbes"
        ),
    ]:

        grid = create_germination_grid(
            group
        )

        germinated_count = int(
            grid.sum()
        )

        germination_percent = (
            germinated_count
            / TOTAL_CELLS
            * 100
        )

        axis.imshow(
            grid,
            vmin=0,
            vmax=1
        )

        for row_index in range(ROWS):

            for column_index in range(COLS):

                label = (
                    "G"
                    if grid[
                        row_index,
                        column_index
                    ] == 1

                    else "-"
                )

                axis.text(
                    column_index,
                    row_index,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8
                )

        axis.set_title(
            f"{treatment}\n"
            f"{germinated_count}/{TOTAL_CELLS} "
            f"cells ({germination_percent:.1f}%)"
        )

        axis.set_xlabel("Column")
        axis.set_ylabel("Row")

        axis.set_xticks(
            range(COLS)
        )

        axis.set_xticklabels(
            range(1, COLS + 1)
        )

        axis.set_yticks(
            range(ROWS)
        )

        axis.set_yticklabels(
            range(1, ROWS + 1)
        )

    figure.suptitle(
        f"Trial 1 Spatial Comparison "
        f"— Day {int(day)}",
        fontsize=16
    )

    plt.tight_layout()

    plt.savefig(
        STEP4_DIR
        / (
            f"first_trial_day_{int(day):02d}_"
            "microbes_vs_no_microbes_spatial_heatmap.png"
        ),
        dpi=300
    )

    plt.show()

C:\Users\rahma\AppData\Local\Temp\ipykernel_28228\3809695676.py:188: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  figure, axes = plt.subplots(


In [33]:
#17 ROW AND COLUMN GERMINATION SUMMARIES

row_column_records = []

for (
    treatment,
    day
), treatment_day_group in cell_df.groupby(
    [
        "treatment",
        "day"
    ]
):

    for row_number, row_group in (
        treatment_day_group.groupby("row")
    ):

        germinated = int(
            row_group[
                "germinated_estimate"
            ]
            .apply(
                boolean_to_integer
            )
            .sum()
        )

        total = len(row_group)

        row_column_records.append(
            {
                "treatment":
                    treatment,

                "day":
                    int(day),

                "axis":
                    "row",

                "axis_number":
                    int(row_number),

                "germinated_cells":
                    germinated,

                "total_cells":
                    total,

                "germination_percent":
                    round(
                        germinated
                        / total
                        * 100,
                        2
                    ),
            }
        )

    for column_number, column_group in (
        treatment_day_group.groupby("col")
    ):

        germinated = int(
            column_group[
                "germinated_estimate"
            ]
            .apply(
                boolean_to_integer
            )
            .sum()
        )

        total = len(column_group)

        row_column_records.append(
            {
                "treatment":
                    treatment,

                "day":
                    int(day),

                "axis":
                    "column",

                "axis_number":
                    int(column_number),

                "germinated_cells":
                    germinated,

                "total_cells":
                    total,

                "germination_percent":
                    round(
                        germinated
                        / total
                        * 100,
                        2
                    ),
            }
        )


row_column_df = pd.DataFrame(
    row_column_records
)

row_column_df.to_csv(
    STEP4_DIR
    / "first_trial_row_column_germination_summary.csv",
    index=False
)

display(
    row_column_df.head(30)
)

print(
    "Row and column summary saved."
)

,treatment,day,axis,axis_number,germinated_cells,total_cells,germination_percent
0,Microbes,1,row,1,6,10,60.00
1,Microbes,1,row,2,1,10,10.00
2,Microbes,1,row,3,2,10,20.00
3,Microbes,1,row,4,6,10,60.00
4,Microbes,1,row,5,4,10,40.00
5,Microbes,1,row,6,4,10,40.00
6,Microbes,1,row,7,3,10,30.00
7,Microbes,1,column,1,1,7,14.29
8,Microbes,1,column,2,1,7,14.29
9,Microbes,1,column,3,5,7,71.43


Row and column summary saved.


In [35]:
#18 DESCRIPTIVE MICROBES VS NO MICROBES COMPARISON

comparison_records = []

for day in sorted(
    cumulative_df["day"].unique()
):

    microbes_row = get_result_row(
        "Microbes",
        day
    )

    no_microbes_row = get_result_row(
        "No Microbes",
        day
    )

    if (
        microbes_row is None
        or no_microbes_row is None
    ):
        continue

    microbes_percent = float(
        microbes_row[
            "cumulative_germination_percent"
        ]
    )

    no_microbes_percent = float(
        no_microbes_row[
            "cumulative_germination_percent"
        ]
    )

    comparison_records.append(
        {
            "day":
                int(day),

            "microbes_germinated_cells":
                int(
                    microbes_row[
                        "cumulative_germinated_cells"
                    ]
                ),

            "no_microbes_germinated_cells":
                int(
                    no_microbes_row[
                        "cumulative_germinated_cells"
                    ]
                ),

            "microbes_percent":
                microbes_percent,

            "no_microbes_percent":
                no_microbes_percent,

            "difference_microbes_minus_no_microbes_percentage_points":
                round(
                    microbes_percent
                    - no_microbes_percent,
                    2
                ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_records
)

comparison_df.to_csv(
    STEP5_DIR
    / "trial1_treatment_comparison_by_day.csv",
    index=False
)


descriptive_records = []

for treatment, treatment_group in (
    cumulative_df.groupby("treatment")
):

    treatment_group = (
        treatment_group
        .sort_values("day")
    )

    observed_days = (
        treatment_group["day"]
        .to_numpy(dtype=float)
    )

    percentages = (
        treatment_group[
            "cumulative_germination_percent"
        ]
        .to_numpy(dtype=float)
    )

    first_day = int(
        observed_days[0]
    )

    final_day = int(
        observed_days[-1]
    )

    first_percent = float(
        percentages[0]
    )

    final_percent = float(
        percentages[-1]
    )

    if final_day != first_day:
        average_daily_increase = (
            final_percent
            - first_percent
        ) / (
            final_day
            - first_day
        )

    else:
        average_daily_increase = 0.0

    observed_auc = float(
        np.trapz(
            percentages,
            observed_days
        )
    )

    descriptive_records.append(
        {
            "treatment":
                treatment,

            "first_observed_day":
                first_day,

            "final_observed_day":
                final_day,

            "first_germination_percent":
                round(
                    first_percent,
                    2
                ),

            "final_germination_percent":
                round(
                    final_percent,
                    2
                ),

            "increase_percentage_points":
                round(
                    final_percent
                    - first_percent,
                    2
                ),

            "average_increase_per_day_percentage_points":
                round(
                    average_daily_increase,
                    2
                ),

            "observed_germination_curve_auc":
                round(
                    observed_auc,
                    2
                ),
        }
    )


descriptive_df = pd.DataFrame(
    descriptive_records
)

descriptive_df.to_csv(
    STEP5_DIR
    / "trial1_descriptive_metrics.csv",
    index=False
)


plt.figure(figsize=(9, 6))

plt.axhline(
    0,
    linewidth=1
)

plt.plot(
    comparison_df["day"],

    comparison_df[
        "difference_microbes_minus_no_microbes_percentage_points"
    ],

    marker="o",
    linewidth=2
)

plt.title(
    "Trial 1: Microbes Minus No Microbes Difference"
)

plt.xlabel("Day")

plt.ylabel(
    "Difference in germination percentage points"
)

plt.xticks(
    comparison_df["day"]
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    STEP5_DIR
    / "trial1_treatment_difference_by_day.png",
    dpi=300
)

plt.show()


final_comparison = (
    comparison_df
    .sort_values("day")
    .iloc[-1]
)

final_day = int(
    final_comparison["day"]
)

final_difference = float(
    final_comparison[
        "difference_microbes_minus_no_microbes_percentage_points"
    ]
)


if final_difference > 0:

    interpretation = (
        f"Microbes had a "
        f"{abs(final_difference):.2f}-percentage-point "
        f"higher automated estimate on Day {final_day}."
    )

elif final_difference < 0:

    interpretation = (
        f"No Microbes had a "
        f"{abs(final_difference):.2f}-percentage-point "
        f"higher automated estimate on Day {final_day}."
    )

else:

    interpretation = (
        f"Both treatments had the same automated "
        f"estimate on Day {final_day}."
    )


findings_lines = [
    "TRIAL 1 — DESCRIPTIVE FINDINGS",
    "",
    f"Final observed day: Day {final_day}",
    "",
    "Microbes:",
    (
        f"- Germinated cells: "
        f"{int(final_comparison['microbes_germinated_cells'])}"
        f"/{TOTAL_CELLS}"
    ),
    (
        f"- Germination estimate: "
        f"{float(final_comparison['microbes_percent']):.2f}%"
    ),
    "",
    "No Microbes:",
    (
        f"- Germinated cells: "
        f"{int(final_comparison['no_microbes_germinated_cells'])}"
        f"/{TOTAL_CELLS}"
    ),
    (
        f"- Germination estimate: "
        f"{float(final_comparison['no_microbes_percent']):.2f}%"
    ),
    "",
    "Descriptive comparison:",
    f"- {interpretation}",
    "- Trial 1 has one tray per treatment.",
    "- This result is descriptive, not a statistical efficacy claim.",
    "- Automated visible-green detection requires visual or manual validation.",
]


findings_text = "\n".join(
    findings_lines
)

findings_path = (
    STEP5_DIR
    / "trial1_key_findings.txt"
)

findings_path.write_text(
    findings_text,
    encoding="utf-8"
)


display(comparison_df)
display(descriptive_df)

print(findings_text)

,day,microbes_germinated_cells,no_microbes_germinated_cells,microbes_percent,no_microbes_percent,difference_microbes_minus_no_microbes_percentage_points
0,1,26,37,37.14,52.86,-15.72
1,3,45,48,64.29,68.57,-4.28
2,4,50,54,71.43,77.14,-5.71


,treatment,first_observed_day,final_observed_day,first_germination_percent,final_germination_percent,increase_percentage_points,average_increase_per_day_percentage_points,observed_germination_curve_auc
0,Microbes,1,4,37.14,71.43,34.29,11.43,169.29
1,No Microbes,1,4,52.86,77.14,24.28,8.09,194.28


TRIAL 1 — DESCRIPTIVE FINDINGS

Final observed day: Day 4

Microbes:
- Germinated cells: 50/70
- Germination estimate: 71.43%

No Microbes:
- Germinated cells: 54/70
- Germination estimate: 77.14%

Descriptive comparison:
- No Microbes had a 5.71-percentage-point higher automated estimate on Day 4.
- Trial 1 has one tray per treatment.
- This result is descriptive, not a statistical efficacy claim.
- Automated visible-green detection requires visual or manual validation.


In [37]:
#19 FINAL TRIAL 1 OUTPUT INDEX

all_output_files = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

output_index_records = []

for path in all_output_files:

    output_index_records.append(
        {
            "relative_path":
                str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),

            "extension":
                path.suffix.lower(),

            "size_kb":
                round(
                    path.stat().st_size
                    / 1024,
                    2
                ),
        }
    )


output_index_df = pd.DataFrame(
    output_index_records
)

output_index_df.to_csv(
    OUTPUT_ROOT
    / "trial1_output_index.csv",
    index=False
)

display(output_index_df)

print(
    f"Created {len(all_output_files)} "
    "Trial 1 output files."
)

print("\nFinal Trial 1 output folder:")
print(OUTPUT_ROOT)

,relative_path,extension,size_kb
0,01_standardized_images\clicked_corner_cell_cen...,.csv,0.64
1,01_standardized_images\diagnostics\microbes_da...,.jpg,7692.34
2,01_standardized_images\diagnostics\microbes_da...,.jpg,817.47
3,01_standardized_images\diagnostics\microbes_da...,.jpg,7171.21
4,01_standardized_images\diagnostics\microbes_da...,.jpg,786.21
...,...,...,...
59,05_descriptive_analysis\trial1_descriptive_met...,.csv,0.29
60,05_descriptive_analysis\trial1_key_findings.txt,.txt,0.48
61,05_descriptive_analysis\trial1_treatment_compa...,.csv,0.23
62,05_descriptive_analysis\trial1_treatment_diffe...,.png,124.81


Created 64 Trial 1 output files.

Final Trial 1 output folder:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\First Trial (Two Trays)
